# Tracking case analysis: best and worst cases
Compare the selected validation run with baseline, then inspect temporal evidence.
Official sequence HOTA/IDF1 come from saved TrackEval results. Local window F1,
error rate and GT match coverage are diagnostic measures, not HOTA/IDF1.
These are development-set results, not the original UA-DETRAC PR-MOTA benchmark.

## 1. Environment and paths
Run from this repository locally or set `UA_DETRAC_PROJECT` on Colab.
Extract dataset and experiment archives first. No detector/tracker rerun is needed.
Change the paths below if your files are elsewhere. Images must be available in
`DATASET/tracking/<sequence>/img1`. Install missing packages with `%pip install
pandas numpy scipy pillow matplotlib ipywidgets` in a separate cell if needed.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path(os.environ.get("UA_DETRAC_PROJECT", Path.cwd())).resolve()
if not (ROOT / "src").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src").is_dir()), ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATASET = Path(os.environ.get("UA_DETRAC_DATASET", ROOT / "outputs/ua_detrac/dataset_v1.1"))
EXPERIMENT = Path(os.environ.get("UA_DETRAC_EXPERIMENT", ROOT / "outputs/tracking/tuning_h_approved_v1"))
OUT = EXPERIMENT / "case_analysis"
print("Project:", ROOT, "\nDataset:", DATASET, "\nExperiment:", EXPERIMENT)

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as DisplayImage
from src.analysis.tracking_cases import (
    BOX, diagnose, window_scores, comparison_frame, export_clip,
)
from src.tracking.evaluation import GT_COLUMNS

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
OUT.mkdir(parents=True, exist_ok=True)
trials = pd.read_csv(EXPERIMENT / "trials.csv")
best_id = json.loads((EXPERIMENT / "best_config.json").read_text())["config_id"]
baseline_id = str(trials.loc[trials.role.eq("baseline"), "config_id"].iloc[0])
RUNS = {"Baseline": EXPERIMENT / "runs" / baseline_id,
        "Best": EXPERIMENT / "runs" / best_id}
metrics = {name: pd.read_csv(path / "evaluation/per_sequence.csv").set_index("sequence")
           for name, path in RUNS.items()}
sequences = metrics["Best"].index.tolist()
metadata = pd.read_csv(DATASET / "metadata/sequences.csv").set_index("sequence")
ranking = metrics["Best"].join(metadata[["sence_weather", "camera_state", "annotation_coverage"]])
ranking["HOTA_gain"] = ranking.HOTA - metrics["Baseline"].HOTA
ranking = ranking.sort_values("HOTA", ascending=False)
display(ranking[["HOTA", "IDF1", "HOTA_gain", "FP", "FN", "IDS",
                 "annotation_coverage", "sence_weather", "camera_state"]].round(2))
ranking.to_csv(OUT / "sequence_ranking.csv")

## 2. Official best/worst sequence ranking
Scores use a 0-100 scale. Counts are integers. HOTA uses thresholds 0.05-0.95;
CLEAR/Identity use IoU 0.7. The table is per sequence; do not average these HOTA
values to replace the official combined score. Weather and camera state provide
context, not evidence of causation. Compare FP as well as improvements in IDs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
order = ranking.index[::-1]
y = np.arange(len(order))
for name, offset, color in [("Baseline", -0.17, "#8796a5"), ("Best", 0.17, "#168c87")]:
    axes[0].barh(y + offset, metrics[name].loc[order, "HOTA"], height=0.32,
                 label=name, color=color)
axes[0].set(yticks=y, yticklabels=order, xlabel="HOTA (0-100)", xlim=(0, 100),
            title="Best to worst: official sequence HOTA")
axes[0].legend()
delta = metrics["Best"].loc[ranking.index, ["HOTA", "IDF1", "MOTA", "AssA", "DetA"]] - metrics["Baseline"].loc[ranking.index, ["HOTA", "IDF1", "MOTA", "AssA", "DetA"]]
limit = max(1, np.abs(delta.to_numpy()).max())
im = axes[1].imshow(delta, cmap="RdYlGn", vmin=-limit, vmax=limit, aspect="auto")
axes[1].set(yticks=range(len(delta)), yticklabels=delta.index,
            xticks=range(len(delta.columns)), xticklabels=delta.columns,
            title="Best minus baseline (score points)")
for i in range(len(delta)):
    for j in range(len(delta.columns)):
        axes[1].text(j, i, f"{delta.iloc[i, j]:+.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=axes[1], label="Score-point change")
fig.tight_layout()
fig.savefig(OUT / "sequence_comparison.png", bbox_inches="tight")
plt.show()

## 3. Reconstruct error evidence and verify counts
This reads canonical GT once, then matches each class separately. The Hungarian
assignment favors the previous evaluated-frame identity, exactly as CLEAR does.
Only XML-declared frames are scored, including explicitly empty XML frames.
Unknown frames remain in the display timeline but are not scored. Ignore-region
union filtering follows the saved protocol: v1 filters predictions only; v2 filters
both GT and predictions. No size, truncation or occlusion filters are added.
This cell may take a few minutes. It reports progress and checks saved counts.

In [ ]:
started = time.perf_counter()
parts = []
for chunk in pd.read_csv(DATASET / "canonical/annotations.csv", usecols=GT_COLUMNS, chunksize=200000):
    parts.append(chunk[chunk.sequence.isin(sequences)])
ground_truth = pd.concat(parts, ignore_index=True)
annotated = pd.read_csv(DATASET / "metadata/annotated_frames.csv")
ignored = pd.read_csv(DATASET / "canonical/ignored_regions.csv")
analysis, all_windows, all_tracks, checks = {}, [], [], []
for sequence in sequences:
    info = metadata.loc[sequence]
    analysis[sequence] = {}
    gt = ground_truth[ground_truth.sequence.eq(sequence)]
    frames = annotated.loc[annotated.sequence.eq(sequence), "frame_id"]
    regions = ignored.loc[ignored.sequence.eq(sequence), BOX].to_numpy(float)
    for name, run in RUNS.items():
        pred = pd.read_csv(run / "tracks" / f"{sequence}.csv")
        protocol = json.loads((run / "evaluation/protocol.json").read_text())
        result = diagnose(gt, pred, frames, int(info.num_frames), regions, info.width, info.height,
                          protocol_version=protocol["version"])
        analysis[sequence][name] = result
        counts = result["frames"][["FP", "FN", "IDS", "Frag"]].sum().astype(int)
        expected = metrics[name].loc[sequence, counts.index].astype(int)
        assert counts.equals(expected), f"Count mismatch: {sequence} / {name}: {counts} != {expected}"
        checks.append({"sequence": sequence, "run": name, **counts.to_dict()})
        result["events"].to_csv(OUT / f"{sequence}_{name.lower()}_events.csv", index=False)
        windows = window_scores(result["frames"])
        windows["sequence"], windows["run"] = sequence, name
        all_windows.append(windows)
        tracks = result["tracks"].copy()
        tracks["sequence"], tracks["run"] = sequence, name
        all_tracks.append(tracks)
    print(f"Verified {sequence}: FP/FN/IDS/Frag agree for both runs", flush=True)
windows = pd.concat(all_windows, ignore_index=True)
tracks = pd.concat(all_tracks, ignore_index=True)
windows.to_csv(OUT / "window_diagnostics.csv", index=False)
tracks.to_csv(OUT / "gt_track_diagnostics.csv", index=False)
pd.DataFrame(checks).to_csv(OUT / "metric_count_checks.csv", index=False)
print(f"Analysis completed in {time.perf_counter() - started:.1f} seconds")

## 4. Best and worst five-second cases
Windows contain 125 original frames at 25 FPS, at least 30 GT observations and
80% annotated frames. Rank by `(FP + FN + IDS) / GT observations` (lower is better).
This avoids selecting empty scenes as easy successes. Windows overlap; they are
inspection candidates, not independent statistical samples. Images show start,
peak-error frame and end for temporal context. Gray shading marks ignore regions.

In [ ]:
eligible = windows[windows.run.eq("Best")].sort_values(["error_rate", "GT"], ascending=[True, False])
if eligible.empty:
    raise ValueError("No eligible windows; lower min_gt or min_coverage explicitly.")
cases = {"Best window": eligible.iloc[0], "Worst window": eligible.iloc[-1]}
pd.DataFrame(cases).T.to_csv(OUT / "selected_cases.csv", index_label="case")
display(pd.DataFrame(cases).T[["sequence", "start", "end", "GT", "FP", "FN", "IDS", "local_f1", "annotation_coverage", "error_rate"]])
for label, case in cases.items():
    sequence = case.sequence
    local = analysis[sequence]["Best"]["frames"].loc[int(case.start):int(case.end)]
    peak = int((local.FP + local.FN + local.IDS).idxmax())
    fig, axes = plt.subplots(3, 1, figsize=(18, 15))
    for ax, frame in zip(axes, [int(case.start), peak, int(case.end)]):
        ax.imshow(comparison_frame(DATASET, sequence, analysis[sequence], frame))
        ax.axis("off")
    fig.suptitle(f"{label}: {sequence}, frames {int(case.start)}-{int(case.end)}; baseline left / best right")
    fig.tight_layout()
    fig.savefig(OUT / (label.replace(" ", "_").lower() + ".png"), bbox_inches="tight")
    plt.show()

## 5. Error timeline and track survival
A shaded interval means no annotation, not zero error. GT track coverage is the
fraction of annotated GT observations matched at IoU 0.7. It is not IDF1.
Require at least 25 observations when selecting good/bad tracks.

In [ ]:
sequence = cases["Worst window"].sequence
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for ax, (name, result) in zip(axes, analysis[sequence].items()):
    table = result["frames"]
    for field, color in [("FP", "#df4444"), ("FN", "#e49a23"), ("IDS", "#c042ba")]:
        ax.plot(table.index, table[field], label=field, color=color, lw=1)
    ax.fill_between(table.index, 0, 1, where=~table.scored, transform=ax.get_xaxis_transform(),
                    color="gray", alpha=0.25, label="Unannotated")
    ax.set(title=f"{sequence} / {name}", ylabel="Count per frame")
    ax.legend(ncol=4)
axes[-1].set_xlabel("Original frame ID")
fig.tight_layout()
fig.savefig(OUT / "error_timeline.png", bbox_inches="tight")
plt.show()
long_tracks = tracks[(tracks.run == "Best") & (tracks.observations >= 25)]
display(long_tracks.sort_values(["coverage", "IDS"], ascending=[False, True]).head(10))
display(long_tracks.sort_values(["coverage", "IDS"], ascending=[True, False]).head(10))

## 6. Interactive synchronized viewer
Select a sequence, original frame and optional class. Baseline is on the left.
Use Next error / Next IDS to avoid scanning thousands of frames manually.
Widgets require a Jupyter frontend; the static figures above work without them.

In [ ]:
import ipywidgets as widgets

# Colab needs its custom widget manager; local Jupyter does not.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

sequence_picker = widgets.Dropdown(options=sequences, value=sequence, description="Sequence")
frame_slider = widgets.IntSlider(value=1, min=1, max=int(metadata.loc[sequence, "num_frames"]),
                                 description="Frame", continuous_update=False, layout=widgets.Layout(width="85%"))
class_picker = widgets.Dropdown(options=[("All classes", None), ("car", 0), ("bus", 1), ("van", 2), ("others", 3)], description="Class")
next_error = widgets.Button(description="Next error")
next_ids = widgets.Button(description="Next IDS")
viewer = widgets.Output()

def refresh(change=None):
    with viewer:
        clear_output(wait=True)
        panel = comparison_frame(DATASET, sequence_picker.value,
                                 analysis[sequence_picker.value], frame_slider.value, class_picker.value)
        panel.thumbnail((1600, 900))
        display(panel)

def change_sequence(change):
    frame_slider.max = int(metadata.loc[sequence_picker.value, "num_frames"])
    frame_slider.value = 1
    refresh()

def jump(kind):
    events = analysis[sequence_picker.value]["Best"]["events"]
    mask = events.kind.eq("IDS") if kind == "IDS" else events.kind.isin(["FP", "FN", "IDS"])
    if class_picker.value is not None:
        mask &= events.class_id.eq(class_picker.value)
    frames = sorted(events.loc[mask, "frame_id"].unique())
    if frames:
        frame_slider.value = next((int(f) for f in frames if f > frame_slider.value), int(frames[0]))

sequence_picker.observe(change_sequence, names="value")
frame_slider.observe(refresh, names="value")
class_picker.observe(refresh, names="value")
next_error.on_click(lambda _: jump("error"))
next_ids.on_click(lambda _: jump("IDS"))
display(widgets.VBox([widgets.HBox([sequence_picker, class_picker, next_error, next_ids]), frame_slider, viewer]))
refresh()

## 7. Identity switch evidence
Inspect the previous matched observation, switch frame and next evaluated frame.
The previous observation can be separated by an unannotated gap: keep original
frame labels when interpreting this example. The plot below shows prediction IDs
assigned to the same GT track; orange crosses indicate missed GT observations.

In [ ]:
sequence = sequence_picker.value
switches = analysis[sequence]["Best"]["events"].query("kind == 'IDS'")
if switches.empty:
    print("No identity switches in this sequence. Choose another sequence above.")
else:
    switch = switches.iloc[0]
    frame = int(switch.frame_id)
    evaluated = analysis[sequence]["Best"]["frames"].query("scored").index
    following = next((int(f) for f in evaluated if f > frame), frame)
    evidence = [int(switch.previous_match_frame), frame, following]
    fig, axes = plt.subplots(3, 1, figsize=(18, 15))
    for ax, f in zip(axes, evidence):
        ax.imshow(comparison_frame(DATASET, sequence, analysis[sequence], f, int(switch.class_id)))
        ax.axis("off")
    fig.suptitle(f"Identity switch: GT {int(switch.gt_id)}, class {int(switch.class_id)}")
    fig.tight_layout()
    fig.savefig(OUT / "identity_switch_evidence.png", bbox_inches="tight")
    plt.show()
    fig, ax = plt.subplots(figsize=(14, 3))
    for name, result in analysis[sequence].items():
        evidence = result["events"]
        evidence = evidence[evidence.gt_id.eq(switch.gt_id) & evidence.class_id.eq(switch.class_id)]
        matched = evidence[evidence.kind.isin(["TP", "IDS"])]
        ax.scatter(matched.frame_id, matched.pred_id, s=12, label=name)
        missed = evidence[evidence.kind.eq("FN")]
        ax.scatter(missed.frame_id, [-1] * len(missed), marker="x", color="orange", s=15)
    ax.set(xlabel="Original frame ID", ylabel="Prediction ID (-1 = FN)", title="Same GT identity across time")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "identity_timeline.png", bbox_inches="tight")
    plt.show()

## Error close-ups: inspect the evidence at readable scale
Show one FP, FN and IDS example when available. The same crop is applied to
baseline and best; labels retain original GT/prediction IDs. Error categories
come from matching, not from an automatic explanation of their physical cause.

In [ ]:
from src.analysis.tracking_cases import draw_frame
from src.tracking.common import find_sequence_images

sequence = sequence_picker.value
result = analysis[sequence]["Best"]
for kind in ["FP", "FN", "IDS"]:
    events = result["events"][result["events"].kind.eq(kind)]
    if events.empty:
        continue
    event = events.iloc[0]
    frame = int(event.frame_id)
    if kind == "FP":
        box = result["pred"].loc[int(event.pred_index), BOX]
    else:
        target = result["gt"]
        box = target.loc[target.frame_id.eq(frame) & target.track_id.eq(event.gt_id)
                         & target.class_id.eq(event.class_id), BOX].iloc[0]
    x, y, w, h = box.to_numpy(float)
    margin = max(60, w, h)
    image_path = find_sequence_images(DATASET, sequence)[frame]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, (name, evidence) in zip(axes, analysis[sequence].items()):
        panel = draw_frame(image_path, evidence, frame, name)
        crop = (max(0, int(x - margin)), max(46, int(y - margin)),
                min(panel.width, int(x + w + margin)), min(panel.height, int(y + h + margin)))
        ax.imshow(panel.crop(crop))
        ax.set_title(name)
        ax.axis("off")
    fig.suptitle(f"{sequence}: {kind}, frame {frame}, class {int(event.class_id)}")
    fig.tight_layout()
    fig.savefig(OUT / f"closeup_{kind.lower()}.png", bbox_inches="tight")
    plt.show()

## 8. Export short animated comparisons
GIFs play in local Jupyter and Colab without an H.264/FFmpeg dependency.
Each five-second clip uses original timing, sampled every two frames (~12.5 FPS).
Set `EXPORT_GIFS = False` to skip exports. Source tracking outputs remain unchanged.

In [ ]:
EXPORT_GIFS = True
if EXPORT_GIFS:
    for label, case in cases.items():
        path = OUT / (label.replace(" ", "_").lower() + ".gif")
        export_clip(DATASET, case.sequence, analysis[case.sequence], case.start, case.end, path)
        display(DisplayImage(filename=str(path)))
print("Analysis artifacts:", OUT)
print("Review both successes and failures before changing parameters; keep test data untouched.")